For each participant and each drawing, compute cosine similarity to their own drawings of the other five sea animal categories (within-subject). Softmax over the six raw similarities yields `draw_probability_{category}`. Output: `data/preprocessed/draw_cossim_toself.csv`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import ast
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax

In [2]:
EMBED_CSV = "../data/embeddings/highres_sketch_embeddings/image_embeddings/draw_embeddings_with_metadata.csv"
OUT_CSV = "../data/preprocessed/draw_cossim_toself.csv"

In [3]:
# Load dataframe 

draw_embeddings = pd.read_csv(EMBED_CSV)

In [4]:
draw_embeddings.head()

,row_id,target_category,embedding,participant_id,age_months
0,/labs/vislearnlab/experiments/birch-draw/data/...,seahorse,"[-0.5459077773548129, 0.95730626101833, -1.915...",AD001,229.0
1,/labs/vislearnlab/experiments/birch-draw/data/...,octopus,"[-0.48118503700169896, -0.2941142848098647, -0...",AD001,229.0
2,/labs/vislearnlab/experiments/birch-draw/data/...,crab,"[-0.023700368010029844, 0.08183493872786367, 0...",AD001,229.0
3,/labs/vislearnlab/experiments/birch-draw/data/...,turtle,"[0.15110634952487648, 0.06575085020781475, 0.0...",AD001,229.0
4,/labs/vislearnlab/experiments/birch-draw/data/...,shark,"[0.7529725162273385, -1.3186576470422475, 0.61...",AD001,229.0


In [5]:
#Define categories

categories = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]


In [6]:

def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    
    if isinstance(x, list):
        return np.array(x, dtype=float)
    
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    # comma-separated Python-list style: "[0.1, -0.2, ...]"
    if "," in x:
        return np.array(ast.literal_eval(x), dtype=float)
    
    # space-separated numpy-array style: "[0.1 -0.2 ...]"
    return np.fromstring(x.strip("[]"), sep=" ")

draw_embeddings["embedding"] = draw_embeddings["embedding"].apply(parse_embedding)

In [7]:
def add_within_subject_draw_similarities(
    draw_embeddings,
    embedding_col="embedding",
    participant_col="participant_id",
    category_col="target_category",
    categories=None
):
    if categories is None:
        categories = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]

    draw_embeddings = draw_embeddings.copy()
    draw_embeddings[category_col] = draw_embeddings[category_col].str.lower().str.strip()

    for cat in categories:
        draw_embeddings[f"draw_cossim_{cat}"] = np.nan
        draw_embeddings[f"draw_probability_{cat}"] = np.nan

    for pid, participant_df in draw_embeddings.groupby(participant_col):
        for idx, row in participant_df.iterrows():
            query_emb = row[embedding_col]

            if not isinstance(query_emb, np.ndarray):
                continue

            comparison_categories = []
            comparison_embeddings = []

            for cat in categories:
                cat_rows = participant_df[participant_df[category_col] == cat]
                cat_embeddings = [
                    emb for row_idx, emb in zip(cat_rows.index, cat_rows[embedding_col])
                    if isinstance(emb, np.ndarray) and row_idx != idx
                ]

                if len(cat_embeddings) == 0:
                    continue

                category_mean = np.mean(np.stack(cat_embeddings), axis=0)
                comparison_categories.append(cat)
                comparison_embeddings.append(category_mean)

            if len(comparison_embeddings) == 0:
                continue

            category_matrix = np.stack(comparison_embeddings)
            sims = cosine_similarity(query_emb.reshape(1, -1), category_matrix).flatten()
            probs = softmax(sims)

            for cat, sim, prob in zip(comparison_categories, sims, probs):
                draw_embeddings.loc[idx, f"draw_cossim_{cat}"] = sim
                draw_embeddings.loc[idx, f"draw_probability_{cat}"] = prob

    return draw_embeddings

In [8]:
draw_embeddings_with_sims = add_within_subject_draw_similarities(draw_embeddings)

prob_cols = [f"draw_probability_{cat}" for cat in categories]

draw_embeddings_with_sims["prob_sum"] = (
    draw_embeddings_with_sims[prob_cols].sum(axis=1, min_count=1)
)

draw_embeddings_with_sims["prob_sum"].describe()

count    6.350000e+02
mean     1.000000e+00
std      8.971515e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: prob_sum, dtype: float64

In [9]:
draw_embeddings_with_sims.head()

,row_id,target_category,embedding,participant_id,age_months,draw_cossim_crab,draw_probability_crab,draw_cossim_octopus,draw_probability_octopus,draw_cossim_seahorse,draw_probability_seahorse,draw_cossim_shark,draw_probability_shark,draw_cossim_turtle,draw_probability_turtle,draw_cossim_whale,draw_probability_whale,prob_sum
0,/labs/vislearnlab/experiments/birch-draw/data/...,seahorse,"[-0.5459077773548129, 0.95730626101833, -1.915...",AD001,229.0,-0.050381,0.195283,0.126743,0.233125,NaN,NaN,-0.085944,0.188460,-0.072934,0.190928,-0.066269,0.192205,1.0
1,/labs/vislearnlab/experiments/birch-draw/data/...,octopus,"[-0.48118503700169896, -0.2941142848098647, -0...",AD001,229.0,-0.095490,0.186853,NaN,NaN,0.126743,0.233353,-0.150697,0.176817,-0.030418,0.199416,-0.009841,0.203562,1.0
2,/labs/vislearnlab/experiments/birch-draw/data/...,crab,"[-0.023700368010029844, 0.08183493872786367, 0...",AD001,229.0,NaN,NaN,-0.095490,0.180589,-0.050381,0.188921,0.034277,0.205612,0.174802,0.236634,-0.053971,0.188244,1.0
3,/labs/vislearnlab/experiments/birch-draw/data/...,turtle,"[0.15110634952487648, 0.06575085020781475, 0.0...",AD001,229.0,0.174802,0.213534,-0.030418,0.173916,-0.072934,0.166677,0.217222,0.222787,NaN,NaN,0.218565,0.223086,1.0
4,/labs/vislearnlab/experiments/birch-draw/data/...,shark,"[0.7529725162273385, -1.3186576470422475, 0.61...",AD001,229.0,0.034277,0.187092,-0.150697,0.155497,-0.085944,0.165899,NaN,NaN,0.217222,0.224651,0.389400,0.266860,1.0


In [10]:
cols_to_drop = ["row_id", "embedding", "prob_sum"]
result = draw_embeddings_with_sims.drop(columns=[c for c in cols_to_drop if c in draw_embeddings_with_sims.columns])

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows → {OUT_CSV}")
print(result.columns.tolist())

Saved 638 rows → ../data/preprocessed/draw_cossim_toself.csv
['target_category', 'participant_id', 'age_months', 'draw_cossim_crab', 'draw_probability_crab', 'draw_cossim_octopus', 'draw_probability_octopus', 'draw_cossim_seahorse', 'draw_probability_seahorse', 'draw_cossim_shark', 'draw_probability_shark', 'draw_cossim_turtle', 'draw_probability_turtle', 'draw_cossim_whale', 'draw_probability_whale']
